# 💳 Home Credit Default Risk - Alternative Credit Scoring (ACS) EDA

## 📌 Mục Tiêu Phân Tích & Định Hướng Alternative Credit Scoring
Notebook này thực hiện Phân Tích Khám Phá Dữ Liệu (**Exploratory Data Analysis - EDA**) chuyên sâu cho bài toán **Alternative Credit Scoring (Đánh giá tín dụng bằng dữ liệu thay thế)** trên tập dữ liệu **Home Credit Default Risk**:
1. **Khai thác dữ liệu thay thế (Alternative Data)**: Đánh giá khả năng nợ xấu cho nhóm khách hàng "Thin-file / Unbanked" dựa trên thuộc tính nhà ở, trình độ học vấn, thâm niên thiết bị, liên lạc và chỉ số rủi ro mạng lưới xã hội.
2. **Đánh giá điểm tín dụng bên thứ 3 (External Alternative Scores)**: Phân tích 3 nguồn điểm rủi ro tổng hợp `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`.
3. **Phân tích biến mục tiêu (`TARGET`)**: Đánh giá mất cân bằng dữ liệu (Default rate = 8.07%).
4. **Thiết kế chỉ số rủi ro tín dụng thay thế (ACS Features)**: `CREDIT_TO_INCOME_RATIO`, `ANNUITY_TO_INCOME_RATIO`, `DAYS_LAST_PHONE_CHANGE`.
5. **Chuẩn bị pipeline huấn luyện & giải thích bằng SHAP** theo quy trình chuẩn trong `AGENTS.md`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 150)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data/raw/home-credit-default-risk')
if not DATA_DIR.exists():
    DATA_DIR = Path('data/raw/home-credit-default-risk')

print('✓ Đã nạp thành công thư viện & đường dẫn dữ liệu Home Credit (Alternative Credit Scoring).')

---
## 1. 📂 Nạp & Tổng Quan Dữ Liệu Bảng Chính (`application_train.csv`)

In [ ]:
app_train = pd.read_csv(DATA_DIR / 'application_train.csv')
print(f'✓ Tổng số bản ghi (rows): {len(app_train):,}')
print(f'✓ Tổng số thuộc tính (features): {app_train.shape[1]}')
app_train.head(5)

---
## 2. 🎯 Phân Tích Biến Mục Tiêu (`TARGET` - Default Rate)
- `TARGET = 0`: Khách hàng hoàn trả khoản vay đúng hạn.
- `TARGET = 1`: Khách hàng vỡ nợ (Default).

In [ ]:
target_counts = app_train['TARGET'].value_counts()
target_rates = app_train['TARGET'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ['Hoàn trả đúng hạn (0)', 'Vỡ nợ / Rủi ro (1)']
sns.barplot(x=labels, y=target_counts.values, ax=axes[0], palette=['#2a9d8f', '#e76f51'], hue=labels, legend=False)
axes[0].set_title('Số Lượng Khách Hàng Theo Target', fontweight='bold')
axes[0].set_ylabel('Số lượng')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points')

axes[1].pie(target_rates, labels=[f'Trả nợ đúng hạn ({target_rates[0]:.1f}%)', f'Vỡ nợ ({target_rates[1]:.1f}%)'],
            autopct='%1.1f%%', startangle=90, colors=['#2a9d8f', '#e76f51'], explode=(0, 0.1),
            textprops={'fontsize': 12, 'weight': 'bold'})
axes[1].set_title('Tỷ Lệ Vỡ Nợ (Default Rate)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'► Tỷ lệ vỡ nợ chung: {target_rates[1]:.2f}% ({target_counts[1]:,} / {len(app_train):,} khách hàng)')

---
## 3. 📱 Phân Tích Dữ Liệu Thay Thế (Alternative Features Analysis)
Khảo sát các thuộc tính phi truyền thống: Thâm niên liên lạc (`DAYS_LAST_PHONE_CHANGE`), xác thực liên lạc (`FLAG_EMP_PHONE`, `FLAG_EMAIL`), loại hình nhà ở (`NAME_HOUSING_TYPE`), và rủi ro mạng lưới xã hội (`DEF_30_CNT_SOCIAL_CIRCLE`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Thâm niên thay đổi số điện thoại (DAYS_LAST_PHONE_CHANGE converted to years)
app_train['PHONE_CHANGE_YEARS'] = abs(app_train['DAYS_LAST_PHONE_CHANGE']) / 365.25
sns.kdeplot(data=app_train, x='PHONE_CHANGE_YEARS', hue='TARGET', common_norm=False, fill=True, ax=axes[0, 0], palette=['#2a9d8f', '#e76f51'])
axes[0, 0].set_title('Thâm Niên Sử Dụng Số Điện Thoại (Năm) vs Target', fontweight='bold')

# Loại hình nhà ở (Housing Type) vs Default Rate
housing_default = app_train.groupby('NAME_HOUSING_TYPE')['TARGET'].mean().sort_values(ascending=False) * 100
sns.barplot(x=housing_default.values, y=housing_default.index, ax=axes[0, 1], palette='Oranges_r', hue=housing_default.index, legend=False)
axes[0, 1].set_title('Tỷ Lệ Vỡ Nợ (%) Theo Loại Hình Nhà Ở', fontweight='bold')
axes[0, 1].set_xlabel('Tỷ lệ vỡ nợ (%)')

# Trình độ học vấn vs Default Rate
edu_default = app_train.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values(ascending=False) * 100
sns.barplot(x=edu_default.values, y=edu_default.index, ax=axes[1, 0], palette='Purples_r', hue=edu_default.index, legend=False)
axes[1, 0].set_title('Tỷ Lệ Vỡ Nợ (%) Theo Trình Độ Học Vấn', fontweight='bold')
axes[1, 0].set_xlabel('Tỷ lệ vỡ nợ (%)')

# Rủi ro mạng lưới xã hội (DEF_30_CNT_SOCIAL_CIRCLE)
sns.boxplot(data=app_train, x='TARGET', y='DEF_30_CNT_SOCIAL_CIRCLE', ax=axes[1, 1], palette=['#2a9d8f', '#e76f51'], hue='TARGET', legend=False)
axes[1, 1].set_title('Số Người Trong Mạng Lưới Xã Hội Vỡ Nợ (30 ngày) vs Target', fontweight='bold')
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(['Trả nợ (0)', 'Vỡ nợ (1)'])
axes[1, 1].set_ylim(-0.5, 5)

plt.tight_layout()
plt.show()

---
## 4. 🌟 Phân Tích Điểm Tín Dụng Bên Thứ Ba (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`)
Đây là các điểm số rủi ro tín dụng tổng hợp từ các đối tác thay thế bên ngoài (Third-party alternative scores).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']):
    sns.kdeplot(data=app_train, x=col, hue='TARGET', common_norm=False, fill=True, ax=axes[i], palette=['#2a9d8f', '#e76f51'])
    axes[i].set_title(f'Phân Phối {col} vs Target', fontweight='bold')

plt.tight_layout()
plt.show()

corrs = app_train[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'TARGET']].corr()['TARGET'].sort_values()
print('Hệ số tương quan Pearson với TARGET:')
print(corrs)

---
## 5. 💡 Feature Engineering Cho Alternative Credit Scoring
Tạo các chỉ số khả năng chi trả thay thế cho khách hàng không có báo cáo tài chính chính thức.

In [ ]:
# Tạo biến phái sinh tài chính thay thế
app_train['CREDIT_TO_INCOME_RATIO'] = app_train['AMT_CREDIT'] / (app_train['AMT_INCOME_TOTAL'] + 1)
app_train['ANNUITY_TO_INCOME_RATIO'] = app_train['AMT_ANNUITY'] / (app_train['AMT_INCOME_TOTAL'] + 1)
app_train['EMPLOYMENT_TO_AGE_RATIO'] = abs(app_train['DAYS_EMPLOYED']) / (abs(app_train['DAYS_BIRTH']) + 1)

print('Hệ số tương quan của các biến phái sinh với TARGET:')
print(app_train[['CREDIT_TO_INCOME_RATIO', 'ANNUITY_TO_INCOME_RATIO', 'EMPLOYMENT_TO_AGE_RATIO', 'TARGET']].corr()['TARGET'])

---
## 6. 🎯 Định Hướng Xây Dựng Mô Hình Alternative Credit Scoring

### 📌 Tóm Tắt Kết Quả EDA:
1. **Mất cân bằng dữ liệu**: Tỷ lệ vỡ nợ **8.07%**. Đánh giá mô hình bằng **ROC-AUC**, **PR-AUC**, **KS Statistic** (Mục tiêu $KS > 40\%$).
2. **Giá trị của Dữ Liệu Thay Thế (Alternative Data)**:
   - `EXT_SOURCE_1/2/3` đóng vai trò quan trọng nhất trong phân tách rủi ro.
   - Thâm niên liên lạc (`PHONE_CHANGE_YEARS`), loại hình nhà ở và mạng lưới xã hội mang lại tín hiệu dự báo mạnh cho nhóm Unbanked.

### 💡 Các Bước Tiếp Theo Theo Quy Trình `AGENTS.md`:
- **Thành lập Baseline Model**: Weight of Evidence (WoE) + Logistic Regression.
- **Mô hình học máy nâng cao**: LightGBM / XGBoost với dữ liệu đã được tổng hợp từ `bureau.csv` và `previous_application.csv`.
- **Giải thích mô hình**: Sử dụng **SHAP values** để minh bạch hóa quyết định cấp tín dụng thay thế.